<a href="https://colab.research.google.com/github/asmaatefomran/generative-ai-tasks/blob/main/Task1_structured_information_extractor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q langchain langchain-openai pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.1/125.1 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.0/570.0 kB 17.9 MB/s eta 0:00:00


In [25]:
!pip install -q langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 12.2 MB/s eta 0:00:00


In [26]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

print("Groq API key loaded successfully.")


Groq API key loaded successfully.


In [27]:
from langchain_groq import ChatGroq
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field, ValidationError
from typing import Optional, List
import json


In [28]:
class CandidateInfo(BaseModel):
    Candidate_name: Optional[str] = None
    Years_of_experience: Optional[float] = None
    Current_role: Optional[str] = None
    Skills: List[str] = Field(default_factory=list)
    Highest_Education: Optional[str] = None


In [32]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)


In [33]:
candidate_text = """
My name is Asmaa Atef. I am currently working as a Senior Python
Developer. I have 6 years of experience in software development.

I have a Bachelor's degree in Computer Science from Cairo University.
My skills include Python, Django, FastAPI, SQL, Git, Docker,
and REST APIs.
"""


In [34]:
prompt = f"""
Extract candidate information from the following text.

Return ONLY valid JSON.

The JSON must contain exactly these fields:

- Candidate_name
- Years_of_experience
- Current_role
- Skills
- Highest_Education

Rules:
1. Use only information explicitly available in the text.
2. Do not invent information.
3. If information is missing, return null.
4. Skills must be a list of strings.
5. Years_of_experience must be a number or null.

Candidate text:

{candidate_text}
"""

response = llm.invoke(prompt)

print(response.content)


{
  "Candidate_name": "Asmaa Atef",
  "Years_of_experience": 6,
  "Current_role": "Senior Python Developer",
  "Skills": [
    "Python",
    "Django",
    "FastAPI",
    "SQL",
    "Git",
    "Docker",
    "REST APIs"
  ],
  "Highest_Education": "Bachelor's degree in Computer Science from Cairo University"
}


In [35]:
try:
    parsed_json = json.loads(response.content)

    print("JSON parsing successful!")
    print(json.dumps(parsed_json, indent=2))

except json.JSONDecodeError as e:
    print("Invalid JSON returned by the model.")
    print(e)


JSON parsing successful!
{
  "Candidate_name": "Asmaa Atef",
  "Years_of_experience": 6,
  "Current_role": "Senior Python Developer",
  "Skills": [
    "Python",
    "Django",
    "FastAPI",
    "SQL",
    "Git",
    "Docker",
    "REST APIs"
  ],
  "Highest_Education": "Bachelor's degree in Computer Science from Cairo University"
}


In [36]:
try:
    candidate = CandidateInfo.model_validate(parsed_json)

    print("Schema validation successful!")

except ValidationError as e:
    print("Schema validation failed:")
    print(e)


Schema validation successful!


In [37]:
try:
    candidate = CandidateInfo.model_validate(parsed_json)

    final_output = candidate.model_dump()

    print(json.dumps(final_output, indent=2))

except ValidationError as e:
    print("Invalid candidate data.")
    print(e)


{
  "Candidate_name": "Asmaa Atef",
  "Years_of_experience": 6.0,
  "Current_role": "Senior Python Developer",
  "Skills": [
    "Python",
    "Django",
    "FastAPI",
    "SQL",
    "Git",
    "Docker",
    "REST APIs"
  ],
  "Highest_Education": "Bachelor's degree in Computer Science from Cairo University"
}
